# Lab 04-01: PyFunc Chain Packaging

**Module:** 04 -- Assembling & Deploying Applications (22% of exam)  
**UI Sections:** Experiments | Models | Serving  
**Estimated Time:** 50--60 minutes  
**Difficulty:** Intermediate-Advanced

---

## What and Why

`mlflow.pyfunc` is how you wrap ANY Python code -- including LangChain chains, custom RAG
pipelines, multi-step agents, and ensemble models -- into a deployable MLflow model. Once
wrapped, the model has a standard interface (`predict(model_input)`) that Databricks Model
Serving knows how to load and execute.

The exam tests your understanding of:
- The `PythonModel` class and its `predict()` method
- Model signatures (input/output schemas)
- How pyfunc connects to the serving layer

Without pyfunc, your chain lives in a notebook and cannot be served as an API. With pyfunc,
it becomes a portable, versioned, deployable artifact.

---

## Mental Model

> **Analogy:** pyfunc is like a shipping container for AI. No matter what is inside -- a
> LangChain chain, custom code, an ensemble of models -- the container has a standard
> interface: `predict(model_input)`. Databricks Serving knows how to load any container.
> It does not care what is inside, only that the interface is standard. Just as global
> shipping works because every port can handle a standard container, Model Serving works
> because every endpoint can handle a standard pyfunc model.

---

## Exam Alert

| Trap (Wrong) | Correct Answer |
|---|---|
| "pyfunc only works with traditional ML models" | pyfunc wraps ANY Python code -- chains, agents, preprocessors, ensembles |
| "Use mlflow.sklearn for LLM chains" | Use `mlflow.pyfunc.PythonModel` for custom GenAI logic |
| "Model signature is optional" | Signature is **required** for serving -- it defines input/output schema |
| "pyfunc models need a training step" | pyfunc has no `fit()` -- it wraps inference-only logic |

---

## Learning Objectives

By the end of this lab you will be able to:

1. Define a custom `mlflow.pyfunc.PythonModel` with a `predict()` method
2. Wrap an LLM call (and a full RAG chain) inside a pyfunc class
3. Define `ModelSignature` with `ColSpec` for input and output schemas
4. Log a pyfunc model with `mlflow.pyfunc.log_model()`
5. Load and test a logged model with `mlflow.pyfunc.load_model()`
6. Register a model to Unity Catalog
7. Explain why pyfunc is the bridge from notebook to serving endpoint

---

## Exam Objectives Covered

| Objective | Tested Here |
|---|---|
| Wrap custom Python logic in mlflow.pyfunc | Steps 1--4 |
| Define model signatures with ColSpec | Step 3 |
| Log and load pyfunc models | Steps 4--5 |
| Register models to Unity Catalog | Step 6 |
| Connect pyfunc to serving endpoints | Step 7 |

---

## Step 1: The pyfunc Interface -- PythonModel Class

At its core, `mlflow.pyfunc.PythonModel` is an abstract class with one required method:
`predict(context, model_input, params=None)`. That is it. Everything else is optional.

### The Class Skeleton

```python
import mlflow

class MyModel(mlflow.pyfunc.PythonModel):
    def predict(self, context, model_input, params=None):
        # Your inference logic here
        return result
```

### Optional Methods

| Method | Purpose | When to Use |
|---|---|---|
| `load_context(context)` | Initialize resources (load files, connect to APIs) | When your model needs artifacts or setup before prediction |
| `predict(context, model_input, params)` | Run inference | Always -- this is the only required method |

### What is `context`?

The `context` parameter provides access to artifacts bundled with the model. For example,
if you include a config file or a retriever index, you access them via
`context.artifacts["my_file"]`.

### What is `model_input`?

By default, `model_input` is a pandas DataFrame. When the model is served via REST API,
the JSON request body is converted to a DataFrame before being passed to `predict()`.

In [ ]:
import mlflow
import pandas as pd

# ---------------------------------------------------------------------------
# The simplest possible pyfunc model: echo the input with a greeting
# ---------------------------------------------------------------------------
class EchoModel(mlflow.pyfunc.PythonModel):
    """A trivial pyfunc model to illustrate the interface."""

    def predict(self, context, model_input, params=None):
        # model_input is a pandas DataFrame by default
        messages = model_input["message"].tolist()
        return [f"Echo: {msg}" for msg in messages]


# Quick local test (no MLflow logging yet)
model = EchoModel()
test_input = pd.DataFrame({"message": ["Hello, pyfunc!", "How does this work?"]})
result = model.predict(context=None, model_input=test_input)
print("Input:", test_input["message"].tolist())
print("Output:", result)


**Expected output:**

```
Input: ['Hello, pyfunc!', 'How does this work?']
Output: ['Echo: Hello, pyfunc!', 'Echo: How does this work?']
```

The key insight: `predict()` receives a DataFrame and returns a list (or DataFrame). This
contract is what makes pyfunc models servable -- the REST API knows how to convert JSON
to/from DataFrames.

---

## Step 2: Wrapping an LLM Call in pyfunc

Now let us wrap a real LLM call. On Databricks, you would call the Foundation Model API
via `openai`-compatible endpoints or `langchain_databricks.ChatDatabricks`. In pyfunc, you
initialize the client in `load_context()` (or `__init__`) and call it in `predict()`.

### Why wrap an LLM call?

- **Preprocessing**: validate inputs, enforce max length, sanitize prompts
- **Postprocessing**: parse structured output, add metadata, filter responses
- **Encapsulation**: the consumer calls `predict()` and does not know or care what LLM
  is behind it -- you can swap models without changing the API

In [ ]:
# ---------------------------------------------------------------------------
# A pyfunc model that wraps an LLM call with pre/post-processing
# ---------------------------------------------------------------------------
class LLMChainModel(mlflow.pyfunc.PythonModel):
    """
    Wraps a Databricks Foundation Model API call in a pyfunc model.
    Includes input validation and output formatting.
    """

    def load_context(self, context):
        """
        Called once when the model is loaded. Use this to initialize
        expensive resources (API clients, loaded artifacts, etc.).
        """
        # In production on Databricks, you would initialize the client:
        # from openai import OpenAI
        # self.client = OpenAI(
        #     base_url="https://<workspace>.databricks.com/serving-endpoints",
        #     api_key=dbutils.secrets.get(scope="genai", key="token")
        # )
        self.model_name = "databricks-meta-llama-3-3-70b-instruct"
        self.max_input_length = 2000  # guard against overly long inputs
        print(f"Model loaded. Using: {self.model_name}")

    def _preprocess(self, question: str) -> str:
        """Validate and clean the input before sending to the LLM."""
        question = question.strip()
        if len(question) == 0:
            raise ValueError("Empty question received.")
        if len(question) > self.max_input_length:
            question = question[:self.max_input_length]
            print(f"  [WARN] Input truncated to {self.max_input_length} chars.")
        return question

    def _postprocess(self, raw_response: str) -> dict:
        """Format the LLM output into a structured response."""
        return {
            "answer": raw_response.strip(),
            "model": self.model_name,
            "char_count": len(raw_response.strip()),
        }

    def predict(self, context, model_input, params=None):
        """
        Main inference method.
        Input: DataFrame with a 'question' column.
        Output: list of dicts with 'answer', 'model', 'char_count'.
        """
        questions = model_input["question"].tolist()
        results = []
        for q in questions:
            clean_q = self._preprocess(q)

            # -- Databricks production code --
            # response = self.client.chat.completions.create(
            #     model=self.model_name,
            #     messages=[
            #         {"role": "system", "content": "You are a helpful assistant."},
            #         {"role": "user", "content": clean_q},
            #     ],
            #     max_tokens=512,
            #     temperature=0.1,
            # )
            # raw = response.choices[0].message.content

            # -- Local simulation --
            raw = f"[Simulated LLM response for: {clean_q[:60]}...]"

            results.append(self._postprocess(raw))
        return results


# Test locally
model = LLMChainModel()
model.load_context(context=None)

test_input = pd.DataFrame({
    "question": [
        "What is MLflow pyfunc?",
        "How do I deploy a model on Databricks?",
    ]
})
results = model.predict(context=None, model_input=test_input)
for r in results:
    print(r)


**Expected output:**

```
Model loaded. Using: databricks-meta-llama-3-3-70b-instruct
{'answer': '[Simulated LLM response for: What is MLflow pyfunc?...]', 'model': 'databricks-meta-llama-3-3-70b-instruct', 'char_count': 52}
{'answer': '[Simulated LLM response for: How do I deploy a model on Databricks?...]', 'model': 'databricks-meta-llama-3-3-70b-instruct', 'char_count': 65}
```

Notice the pattern:
- `load_context()` runs once (initializes the client)
- `predict()` runs per request (processes each question)
- Pre/post-processing lives inside the class -- invisible to the caller

---

## Step 3: Defining ModelSignature with ColSpec

A `ModelSignature` tells MLflow (and Databricks Serving) the **exact shape** of the data
your model expects and produces. Without a signature, the serving endpoint cannot validate
requests or generate API documentation.

### Signature Components

| Component | What It Defines | Example |
|---|---|---|
| `inputs` | Schema of the input DataFrame | `ColSpec("string", "question")` |
| `outputs` | Schema of the output | `ColSpec("string", "answer")` |
| `params` | Optional inference parameters | `ParamSpec("temperature", "double", 0.1)` |

### Why ColSpec?

`ColSpec` defines a named, typed column. Think of it as declaring the API contract:
"This model accepts a column called `question` of type `string` and returns a column
called `answer` of type `string`."

> **Analogy:** A ModelSignature is like a restaurant menu. It tells the customer (API
> caller) exactly what they can order (input schema) and what they will receive (output
> schema). Without the menu, every order is a guessing game.

In [ ]:
from mlflow.models import infer_signature
from mlflow.models.signature import ModelSignature
from mlflow.types.schema import Schema, ColSpec, ParamSchema, ParamSpec

# ---------------------------------------------------------------------------
# Method 1: Explicitly define the signature with ColSpec
# (Preferred for production -- you control every detail)
# ---------------------------------------------------------------------------
input_schema = Schema([
    ColSpec("string", "question"),    # Input column: the user's question
])

output_schema = Schema([
    ColSpec("string", "answer"),      # Output column: the model's answer
])

param_schema = ParamSchema([
    ParamSpec("temperature", "double", 0.1),   # Optional inference parameter
    ParamSpec("max_tokens", "long", 512),       # Optional inference parameter
])

explicit_signature = ModelSignature(
    inputs=input_schema,
    outputs=output_schema,
    params=param_schema,
)

print("Explicit signature:")
print(explicit_signature)
print()

# ---------------------------------------------------------------------------
# Method 2: Infer the signature from sample data
# (Quick and convenient, but less control)
# ---------------------------------------------------------------------------
sample_input = pd.DataFrame({"question": ["What is pyfunc?"]})
sample_output = pd.DataFrame({"answer": ["pyfunc is a packaging format."]})

inferred_signature = infer_signature(sample_input, sample_output)

print("Inferred signature:")
print(inferred_signature)


**Expected output:**

```
Explicit signature:
inputs: 
  ['question': string (required)]
outputs: 
  ['answer': string (required)]
params: 
  ['temperature': double (default: 0.1), 'max_tokens': long (default: 512)]

Inferred signature:
inputs: 
  ['question': string (required)]
outputs: 
  ['answer': string (required)]
```

### Exam Tip: Explicit vs. Inferred Signatures

| Approach | Pros | Cons |
|---|---|---|
| Explicit (ColSpec) | Full control, includes params, self-documenting | More code to write |
| Inferred | Quick, less code | May miss params, less precise types |

For the exam, know that `ColSpec` is used for explicit signatures and that the signature
defines the **contract** between caller and model.

---

## Step 4: Logging with mlflow.pyfunc.log_model()

Logging is the act of saving your model (class + artifacts + signature + dependencies) to
an MLflow run. After logging, the model is a versioned artifact that can be loaded,
registered, and deployed.

### Key Parameters of `log_model()`

| Parameter | Purpose | Example |
|---|---|---|
| `artifact_path` | Name for the model artifact directory | `"rag_chain"` |
| `python_model` | Instance of your PythonModel class | `LLMChainModel()` |
| `signature` | Input/output/params schema | `explicit_signature` |
| `conda_env` | Python dependencies (auto-detected or manual) | `{"dependencies": [...]}` |
| `code_paths` | Additional Python files to include | `["./utils.py"]` |
| `artifacts` | Files bundled with the model | `{"config": "config.yaml"}` |
| `pip_requirements` | Simpler alternative to conda_env | `["openai>=1.0", "langchain"]` |
| `input_example` | Sample input for documentation | `pd.DataFrame({"question": ["test"]})` |

### What Happens Behind the Scenes

When you call `log_model()`, MLflow:
1. Serializes your `PythonModel` class using `cloudpickle`
2. Saves the conda environment (or pip requirements)
3. Saves the signature as metadata
4. Saves any artifacts or code_paths
5. Writes everything to the MLflow tracking server (or local artifact store)

In [ ]:
# ---------------------------------------------------------------------------
# Log the pyfunc model to MLflow
# ---------------------------------------------------------------------------
import mlflow

# Set experiment (on Databricks, this maps to a workspace folder)
# mlflow.set_experiment("/Users/<your-email>/04-01-pyfunc-chain")  # Databricks
mlflow.set_experiment("04-01-pyfunc-chain")  # Local

with mlflow.start_run(run_name="llm-chain-pyfunc") as run:
    # Define the conda environment (dependencies the model needs)
    conda_env = {
        "channels": ["defaults"],
        "dependencies": [
            "python=3.11",
            "pip",
            {"pip": [
                "mlflow>=2.12",
                "openai>=1.0",
                "pandas",
            ]},
        ],
        "name": "llm_chain_env",
    }

    # Log the model
    model_info = mlflow.pyfunc.log_model(
        artifact_path="llm_chain",
        python_model=LLMChainModel(),
        signature=explicit_signature,
        conda_env=conda_env,
        input_example=pd.DataFrame({"question": ["What is MLflow?"]}),
    )

    print(f"Run ID:     {run.info.run_id}")
    print(f"Model URI:  {model_info.model_uri}")
    print(f"Artifact:   {model_info.artifact_path}")

    # Log some tags for discoverability
    mlflow.set_tag("model_type", "llm_chain")
    mlflow.set_tag("llm", "databricks-meta-llama-3-3-70b-instruct")


**Expected output:**

```
Run ID:     a1b2c3d4e5f6...
Model URI:  runs:/a1b2c3d4e5f6.../llm_chain
Artifact:   llm_chain
```

### UI Navigation: View the Logged Model

**UI -->** Left nav --> **Experiments** --> select `04-01-pyfunc-chain` --> click the run
--> **Artifacts** tab --> expand `llm_chain` folder

You will see:
- `MLmodel` -- metadata file with signature, flavors, env
- `conda.yaml` -- the conda environment
- `python_model.pkl` -- your serialized PythonModel class
- `requirements.txt` -- pip requirements extracted from conda env

---

## Step 5: Loading and Testing Locally

After logging, you should always test your model by loading it back. This verifies that:
1. Serialization worked (the class was pickled correctly)
2. The signature is enforced (bad inputs are rejected)
3. `load_context()` runs successfully (resources initialize)

### Two Ways to Load

| Method | URI Format | Use Case |
|---|---|---|
| `mlflow.pyfunc.load_model(model_uri)` | `runs:/<run_id>/artifact_path` | Load from a specific run |
| `mlflow.pyfunc.load_model(model_uri)` | `models:/<model_name>/<version>` | Load from model registry |

In [ ]:
# ---------------------------------------------------------------------------
# Load the model back and test it
# ---------------------------------------------------------------------------
loaded_model = mlflow.pyfunc.load_model(model_info.model_uri)

# Test with valid input
test_df = pd.DataFrame({
    "question": [
        "Explain pyfunc in one sentence.",
        "What is the difference between pyfunc and sklearn flavor?",
        "How do I deploy an MLflow model?",
    ]
})

predictions = loaded_model.predict(test_df)
print("Predictions:")
for i, pred in enumerate(predictions):
    print(f"  Q{i+1}: {test_df['question'].iloc[i]}")
    print(f"  A{i+1}: {pred}")
    print()


**Expected output:**

```
Model loaded. Using: databricks-meta-llama-3-3-70b-instruct
Predictions:
  Q1: Explain pyfunc in one sentence.
  A1: {'answer': '[Simulated LLM response for: Explain pyfunc in one sentence....]', 'model': 'databricks-meta-llama-3-3-70b-instruct', 'char_count': 58}

  Q2: What is the difference between pyfunc and sklearn flavor?
  A2: {'answer': '[Simulated LLM response for: What is the difference between pyfunc and sklearn fl...]', 'model': 'databricks-meta-llama-3-3-70b-instruct', 'char_count': 84}

  Q3: How do I deploy an MLflow model?
  A3: {'answer': '[Simulated LLM response for: How do I deploy an MLflow model?...]', 'model': 'databricks-meta-llama-3-3-70b-instruct', 'char_count': 59}
```

The loaded model behaves identically to the original class. This is the whole point of
pyfunc: serialize once, load anywhere.

In [ ]:
# ---------------------------------------------------------------------------
# Demonstrate signature enforcement
# ---------------------------------------------------------------------------
# The signature says the model expects a 'question' column.
# What happens if we pass the wrong column name?

try:
    bad_input = pd.DataFrame({"wrong_column": ["This should fail"]})
    loaded_model.predict(bad_input)
except Exception as e:
    print(f"Signature enforcement caught the error!")
    print(f"Error type: {type(e).__name__}")
    print(f"Error message: {e}")


**Expected output:**

```
Signature enforcement caught the error!
Error type: MlflowException
Error message: Model is missing inputs ['question'].
```

This is why signatures matter for serving: they catch invalid requests before they reach
your model code, returning a clear 400 error to the API caller.

---

## Step 6: Registering to Unity Catalog

Registering a model to Unity Catalog makes it a **governed asset** with:
- **Three-level namespace**: `catalog.schema.model_name`
- **Version history**: every registration creates a new version
- **Access control**: GRANT/REVOKE permissions on the model
- **Lineage**: track which data and code produced each version

### The Registration URI

On Databricks with Unity Catalog, the model URI uses the format:
`models:/catalog.schema.model_name`

### Key Difference from Legacy Registry

| Feature | Legacy Workspace Registry | Unity Catalog Registry |
|---|---|---|
| Namespace | `models:/model_name` | `models:/catalog.schema.model_name` |
| Governance | Workspace-level | Organization-level |
| Cross-workspace | No | Yes |
| Access control | Basic | Fine-grained (SQL GRANT/REVOKE) |

In [ ]:
# ---------------------------------------------------------------------------
# Register the model to Unity Catalog
# ---------------------------------------------------------------------------
# On Databricks with Unity Catalog enabled:
#
# import mlflow
# mlflow.set_registry_uri("databricks-uc")  # Use Unity Catalog registry
#
# MODEL_NAME = "main.genai_labs.llm_chain_model"
#
# # Register from the run
# result = mlflow.register_model(
#     model_uri=model_info.model_uri,
#     name=MODEL_NAME,
# )
#
# print(f"Registered: {result.name}")
# print(f"Version:    {result.version}")
# print(f"Status:     {result.status}")
#
# # Add a description and tags
# from mlflow import MlflowClient
# client = MlflowClient()
# client.update_model_version(
#     name=MODEL_NAME,
#     version=result.version,
#     description="LLM chain with pre/post-processing, packaged as pyfunc.",
# )

MODEL_NAME = "main.genai_labs.llm_chain_model"

print("Registration command (run on Databricks):")
print()
print("  import mlflow")
print("  mlflow.set_registry_uri('databricks-uc')")
print()
print(f"  result = mlflow.register_model(")
print(f"      model_uri='{model_info.model_uri}',")
print(f"      name='{MODEL_NAME}',")
print(f"  )")
print()
print(f"  # Result: {MODEL_NAME} version 1")
print()
print("UI Navigation:")
print("  **UI -->** Left nav --> **Models** --> search for 'llm_chain_model'")
print("  You will see all versions, their status, and lineage.")


**Expected output:**

```
Registration command (run on Databricks):

  import mlflow
  mlflow.set_registry_uri('databricks-uc')

  result = mlflow.register_model(
      model_uri='runs:/a1b2c3d4.../llm_chain',
      name='main.genai_labs.llm_chain_model',
  )

  # Result: main.genai_labs.llm_chain_model version 1
```

### UI Navigation: View the Registered Model

**UI -->** Left nav --> **Models** --> search for `llm_chain_model` --> click the model

You will see:
- **Versions** tab: list of all model versions with their source runs
- **Details** tab: description, tags, creation date
- **Lineage** tab: which experiment run produced this version
- **Permissions** tab: who can access this model (GRANT/REVOKE)

---

## Step 7: A Complete RAG Chain in pyfunc

Now let us put it all together: a pyfunc model that wraps a RAG chain (retriever + prompt +
LLM). This is what you would deploy to a serving endpoint in production.

### Architecture Inside the pyfunc

```
User Question
    |
    v
[Preprocessing] -- validate, clean, truncate
    |
    v
[Retriever] -- query Vector Search index for relevant chunks
    |
    v
[Prompt Builder] -- combine question + retrieved context into a prompt
    |
    v
[LLM Call] -- send prompt to Foundation Model API
    |
    v
[Postprocessing] -- format output, add sources, enforce guardrails
    |
    v
Structured Response
```

In [ ]:
# ---------------------------------------------------------------------------
# A full RAG chain wrapped in pyfunc
# ---------------------------------------------------------------------------
class RAGChainModel(mlflow.pyfunc.PythonModel):
    """
    A complete RAG pipeline packaged as a pyfunc model.
    Retrieves context from a vector index, builds a prompt, and calls an LLM.
    """

    def load_context(self, context):
        """Initialize the retriever, LLM client, and configuration."""
        # In production on Databricks:
        # from databricks.vector_search.client import VectorSearchClient
        # from openai import OpenAI
        #
        # self.vsc = VectorSearchClient()
        # self.index = self.vsc.get_index(
        #     endpoint_name="genai_vs_endpoint",
        #     index_name="main.genai_labs.document_chunks_index"
        # )
        # self.client = OpenAI(...)

        self.model_name = "databricks-meta-llama-3-3-70b-instruct"
        self.system_prompt = (
            "You are a helpful assistant. Answer the user's question based "
            "on the provided context. If the context does not contain the "
            "answer, say 'I don't have enough information to answer that.'"
        )
        print("RAG Chain loaded: retriever + prompt builder + LLM")

    def _retrieve(self, query: str, num_results: int = 3) -> list:
        """Query the vector search index for relevant chunks."""
        # Production:
        # results = self.index.similarity_search(
        #     query_text=query,
        #     columns=["chunk_text", "source_doc"],
        #     num_results=num_results,
        # )
        # return results.get("result", {}).get("data_array", [])

        # Simulation
        return [
            ["Databricks is a unified analytics platform built on Spark.", "databricks_overview.txt"],
            ["Unity Catalog provides fine-grained access control.", "databricks_overview.txt"],
            ["Delta Lake provides ACID transactions and schema enforcement.", "databricks_overview.txt"],
        ]

    def _build_prompt(self, question: str, context_chunks: list) -> str:
        """Combine retrieved context with the user's question."""
        context_text = "\n---\n".join([chunk[0] for chunk in context_chunks])
        return (
            f"Context:\n{context_text}\n\n"
            f"Question: {question}\n\n"
            f"Answer based on the context above:"
        )

    def predict(self, context, model_input, params=None):
        """Run the full RAG pipeline: retrieve -> build prompt -> call LLM."""
        questions = model_input["question"].tolist()
        results = []

        for q in questions:
            # Step 1: Retrieve relevant context
            chunks = self._retrieve(q)

            # Step 2: Build augmented prompt
            prompt = self._build_prompt(q, chunks)

            # Step 3: Call LLM (simulated)
            # Production:
            # response = self.client.chat.completions.create(
            #     model=self.model_name,
            #     messages=[
            #         {"role": "system", "content": self.system_prompt},
            #         {"role": "user", "content": prompt},
            #     ],
            # )
            # answer = response.choices[0].message.content
            answer = f"[Simulated RAG answer using {len(chunks)} retrieved chunks]"

            # Step 4: Format the response
            results.append({
                "answer": answer,
                "sources": [c[1] for c in chunks],
                "num_chunks_used": len(chunks),
                "model": self.model_name,
            })

        return results


# Test the RAG chain locally
rag_model = RAGChainModel()
rag_model.load_context(context=None)

test_input = pd.DataFrame({"question": ["What is Unity Catalog?"]})
rag_results = rag_model.predict(context=None, model_input=test_input)
for r in rag_results:
    for k, v in r.items():
        print(f"  {k}: {v}")


**Expected output:**

```
RAG Chain loaded: retriever + prompt builder + LLM
  answer: [Simulated RAG answer using 3 retrieved chunks]
  sources: ['databricks_overview.txt', 'databricks_overview.txt', 'databricks_overview.txt']
  num_chunks_used: 3
  model: databricks-meta-llama-3-3-70b-instruct
```

In [ ]:
# ---------------------------------------------------------------------------
# Log the RAG chain model with a complete signature
# ---------------------------------------------------------------------------
rag_input_schema = Schema([
    ColSpec("string", "question"),
])

rag_output_schema = Schema([
    ColSpec("string", "answer"),
])

rag_signature = ModelSignature(
    inputs=rag_input_schema,
    outputs=rag_output_schema,
)

with mlflow.start_run(run_name="rag-chain-pyfunc") as run:
    rag_model_info = mlflow.pyfunc.log_model(
        artifact_path="rag_chain",
        python_model=RAGChainModel(),
        signature=rag_signature,
        pip_requirements=[
            "mlflow>=2.12",
            "openai>=1.0",
            "databricks-vectorsearch",
            "pandas",
        ],
        input_example=pd.DataFrame({"question": ["What is Databricks?"]}),
    )

    mlflow.set_tag("model_type", "rag_chain")
    mlflow.set_tag("retriever", "vector_search_delta_sync")
    mlflow.set_tag("llm", "databricks-meta-llama-3-3-70b-instruct")

    print(f"RAG chain logged.")
    print(f"  Run ID:    {run.info.run_id}")
    print(f"  Model URI: {rag_model_info.model_uri}")


**Expected output:**

```
RAG chain logged.
  Run ID:    f7e8d9c0b1a2...
  Model URI: runs:/f7e8d9c0b1a2.../rag_chain
```

### Why pyfunc Matters for Deployment

This is the bridge from notebook to production:

```
Notebook Development          Deployment
-------------------          ----------
RAGChainModel class   --->   mlflow.pyfunc.log_model()
                             mlflow.register_model()
                             Create Serving Endpoint
                             POST /serving-endpoints/.../invocations
```

The serving endpoint loads your pyfunc model, calls `load_context()` once, then routes
every API request through `predict()`. The caller sends JSON, it is converted to a
DataFrame, and your `predict()` method handles the rest.

---

## Why pyfunc Is the Bridge to Serving

### The Deployment Sequence

```
1. Develop your model/chain in a notebook
2. Wrap it in a PythonModel class (pyfunc)
3. Log with mlflow.pyfunc.log_model()
4. Register to Unity Catalog with mlflow.register_model()
5. Create a Serving Endpoint pointing to the registered model
6. Clients call the REST API -- pyfunc.predict() handles every request
```

### What Serving Does with Your pyfunc

When you create a serving endpoint for a pyfunc model, Databricks:

1. **Provisions compute** -- spins up container(s) with your conda/pip environment
2. **Loads the model** -- calls `load_context()` to initialize resources
3. **Serves requests** -- routes POST requests to `predict()`, converting JSON to DataFrame
4. **Enforces signature** -- validates input against your ModelSignature before calling predict
5. **Scales** -- auto-scales containers based on traffic

### The Standard Interface

| REST API Input | pyfunc Receives | pyfunc Returns | REST API Output |
|---|---|---|---|
| JSON `{"dataframe_split": {"columns": ["question"], "data": [["What is X?"]]}}` | `pd.DataFrame({"question": ["What is X?"]})` | `[{"answer": "X is..."}]` | JSON `{"predictions": [{"answer": "X is..."}]}` |

In [ ]:
# ---------------------------------------------------------------------------
# Quick-reference: the full pyfunc lifecycle in one cell
# ---------------------------------------------------------------------------
print("=" * 70)
print("PYFUNC LIFECYCLE -- QUICK REFERENCE")
print("=" * 70)
print()
print("1. DEFINE the model class:")
print("   class MyModel(mlflow.pyfunc.PythonModel):")
print("       def load_context(self, context): ...")
print("       def predict(self, context, model_input, params=None): ...")
print()
print("2. DEFINE the signature:")
print("   from mlflow.types.schema import Schema, ColSpec")
print("   signature = ModelSignature(")
print("       inputs=Schema([ColSpec('string', 'question')]),")
print("       outputs=Schema([ColSpec('string', 'answer')]),")
print("   )")
print()
print("3. LOG the model:")
print("   with mlflow.start_run():")
print("       mlflow.pyfunc.log_model(")
print("           artifact_path='my_model',")
print("           python_model=MyModel(),")
print("           signature=signature,")
print("           pip_requirements=[...],")
print("       )")
print()
print("4. LOAD and test:")
print("   loaded = mlflow.pyfunc.load_model('runs:/<id>/my_model')")
print("   loaded.predict(pd.DataFrame({'question': ['test']}))")
print()
print("5. REGISTER to Unity Catalog:")
print("   mlflow.set_registry_uri('databricks-uc')")
print("   mlflow.register_model('runs:/<id>/my_model', 'catalog.schema.name')")
print()
print("6. SERVE:")
print("   Create endpoint pointing to the registered model version.")
print("   POST /serving-endpoints/<name>/invocations")
print("=" * 70)


---

## Exam Tips

| # | Tip | Why It Matters |
|---|---|---|
| 1 | pyfunc wraps ANY Python code -- not just ML models | The exam will try to trick you with "pyfunc is only for sklearn/XGBoost" |
| 2 | `predict(context, model_input, params)` is the ONLY required method | Know the method signature cold |
| 3 | `load_context()` runs once at model load time | Use it for expensive initialization (API clients, file loading) |
| 4 | ModelSignature is required for serving | Without it, the endpoint cannot validate requests |
| 5 | `ColSpec("string", "question")` defines a named, typed column | Know the ColSpec syntax for the exam |
| 6 | Unity Catalog uses `models:/catalog.schema.model_name` | Three-level namespace, not just `models:/name` |
| 7 | `conda_env` or `pip_requirements` ensures reproducibility | The model runs in the exact environment you specified |
| 8 | `code_paths` bundles additional Python files with the model | Use when your model imports from local modules |

---

## Key Takeaways

### The pyfunc Interface
- `mlflow.pyfunc.PythonModel` is the base class for all custom pyfunc models
- `predict(context, model_input, params)` is the only required method
- `load_context(context)` initializes expensive resources (API clients, loaded files)
- The model receives a pandas DataFrame and returns a list or DataFrame

### Model Signature
- Defines the API contract: what inputs are expected, what outputs are produced
- Built with `ColSpec` for explicit schemas or `infer_signature()` for auto-detection
- Required for serving -- enables input validation and API documentation

### Logging and Registration
- `mlflow.pyfunc.log_model()` saves the model to an MLflow run
- `mlflow.register_model()` promotes it to Unity Catalog as a governed asset
- The registered model can then be deployed to a serving endpoint

### The Deployment Bridge
- pyfunc is the universal packaging format for Databricks Model Serving
- Any Python logic can be wrapped: LLM calls, RAG chains, agents, ensembles
- The serving endpoint handles the REST API layer; your `predict()` handles the logic

---

## Next Lab

**Lab 04-02: Vector Search Index** -- Now that you know how to package a model, you need
a retriever to feed it context. Next, we build Vector Search indexes (Delta Sync and
Direct Vector Access) that power the retrieval step in your RAG chain.